# Matemáticas de la Inteligencia Artificial
## Sesión 12 — Self-attention: del producto escalar a una red dinámica de contexto

[![Abrir en Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CuentosCuanticos/matematicas-ia/blob/main/12_self_attention/laboratorio.ipynb)

**Pregunta de la sesión:** ¿cómo puede cada token decidir qué otras posiciones son relevantes para construir su representación contextual?

Construiremos **una cabeza de self-attention causal desde cero**, sin `nn.MultiheadAttention` ni capas de Transformer preconstruidas.

Recorrido matemático:

\[
X\longrightarrow Q,K,V
\longrightarrow \frac{QK^T}{\sqrt{d_k}}
\longrightarrow \text{máscara causal}
\longrightarrow \mathrm{softmax}
\longrightarrow A(X)V.
\]

El objetivo no es memorizar una fórmula: es poder explicar **qué objeto matemático representa cada tensor**, qué dimensiones tiene y por qué cada operación es necesaria.

In [ ]:
import math, random, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 12
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| dispositivo:', device)

## 1. Diccionario matemática ↔ código

Trabajaremos con una secuencia de longitud \(T\) y embeddings de dimensión \(d_{\rm model}\).

| Matemática | PyTorch | Dimensión |
|---|---|---|
| \(X\) | `X` | \((B,T,d_{\rm model})\) |
| \(Q=XW_Q\) | `Q` | \((B,T,d_k)\) |
| \(K=XW_K\) | `K` | \((B,T,d_k)\) |
| \(V=XW_V\) | `Vv` | \((B,T,d_v)\) |
| \(QK^T\) | `Q @ K.transpose(-2,-1)` | \((B,T,T)\) |
| \(A\) | `softmax(scores, dim=-1)` | \((B,T,T)\) |
| \(Z=AV\) | `A @ Vv` | \((B,T,d_v)\) |

**Convención:** \(A_{ij}\) es el peso con el que la posición \(i\) recibe información de la posición \(j\). En la lectura como grafo, la arista es \(j\to i\).

## 2. De tokens a vectores

Usaremos una secuencia muy pequeña para poder inspeccionar todos los tensores. El embedding no es todavía contextual: cada token obtiene una fila de una matriz aprendible \(E\).

In [ ]:
tokens = ['la', 'masa', 'curva', 'el', 'espacio', 'tiempo', 'luz']
token_a_id = {t:i for i,t in enumerate(tokens)}
id_a_token = {i:t for t,i in token_a_id.items()}

frase = ['la', 'masa', 'curva', 'el', 'espacio']
ids = torch.tensor([[token_a_id[t] for t in frase]], dtype=torch.long)

d_model = 8
embedding = nn.Embedding(len(tokens), d_model)

# TODO 1: obtén X aplicando el embedding a ids.
X = ...

print('ids:', ids.shape)
print('X  :', X.shape)
assert X.shape == (1, len(frase), d_model)

## 3. Queries, keys y values

Una misma representación \(x_i\) debe desempeñar tres papeles:

- **query** \(q_i\): qué información busca la posición \(i\);
- **key** \(k_j\): cómo se identifica la posición \(j\) para ser comparada;
- **value** \(v_j\): qué información transmite \(j\) si recibe peso.

Introducimos tres transformaciones aprendibles:

\[
Q=XW_Q,\qquad K=XW_K,\qquad V=XW_V.
\]

In [ ]:
d_k = 6
d_v = 5

Wq = nn.Linear(d_model, d_k, bias=False)
Wk = nn.Linear(d_model, d_k, bias=False)
Wv = nn.Linear(d_model, d_v, bias=False)

# TODO 2: calcula Q, K y Vv.
Q = ...
K = ...
Vv = ...

print('Q:', Q.shape, 'K:', K.shape, 'V:', Vv.shape)
assert Q.shape == K.shape == (1, len(frase), d_k)
assert Vv.shape == (1, len(frase), d_v)

## 4. Todas las compatibilidades de una sola vez

Queremos

\[
s_{ij}=q_i^T k_j.
\]

El producto matricial \(QK^T\) calcula simultáneamente todos esos productos escalares.

In [ ]:
# TODO 3: construye la matriz de scores sin escalar.
scores_raw = ...

i, j = 2, 1
# TODO 4: calcula a mano q_i^T k_j usando torch.dot.
score_individual = ...

print('matriz:', scores_raw[0, i, j].item())
print('individual:', score_individual.item())

assert scores_raw.shape == (1, len(frase), len(frase))
assert torch.allclose(scores_raw[0, i, j], score_individual)

## 5. ¿Por qué dividir por \(\sqrt{d_k}\)?

Si las componentes de \(q\) y \(k\) son independientes, centradas y de varianza unidad,

\[
q^Tk=\sum_{r=1}^{d_k}q_rk_r,
\qquad
\operatorname{Var}(q^Tk)=d_k.
\]

Por tanto, la desviación típica crece como \(\sqrt{d_k}\). El escalado

\[
\frac{q^Tk}{\sqrt{d_k}}
\]

mantiene una escala típica de orden uno y evita saturar innecesariamente softmax.

In [ ]:
dims = [4, 16, 64, 256]
tabla = []

for d in dims:
    q = torch.randn(5000, d)
    k = torch.randn(5000, d)
    raw = (q*k).sum(dim=1)
    # TODO 5: divide por la escala correcta.
    scaled = ...
    tabla.append((d, raw.std().item(), scaled.std().item()))

print('d_k | std(q·k) | std((q·k)/sqrt(d_k))')
for fila in tabla:
    print(f'{fila[0]:3d} | {fila[1]:8.3f} | {fila[2]:8.3f}')

## 6. De scores a pesos de atención

Para una fila fija \(i\), softmax produce

\[
\alpha_{ij}
=
\frac{\exp(s_{ij})}{\sum_\ell \exp(s_{i\ell})},
\]

de modo que

\[
\alpha_{ij}\ge 0,
\qquad
\sum_j\alpha_{ij}=1.
\]

La normalización se hace sobre la dimensión de las **keys**: en PyTorch, la última dimensión.

In [ ]:
# TODO 6: escala scores_raw y aplica softmax por filas.
scores = ...
A_full = ...

print(A_full[0])
print('sumas de filas:', A_full[0].sum(dim=-1))

assert torch.all(A_full >= 0)
assert torch.allclose(A_full.sum(dim=-1),
                      torch.ones_like(A_full.sum(dim=-1)),
                      atol=1e-6)

## 7. Atención = combinación ponderada de values

La salida de la posición \(i\) es

\[
z_i=\sum_j\alpha_{ij}v_j.
\]

Como los \(\alpha_{ij}\) son no negativos y suman uno, \(z_i\) es una combinación convexa de los values.

In [ ]:
# TODO 7: calcula Z_full.
Z_full = ...

i = 3
# TODO 8: reconstruye z_i explícitamente como suma ponderada.
z_manual = ...

print('producto matricial:', Z_full[0, i])
print('suma explícita    :', z_manual)

assert torch.allclose(Z_full[0, i], z_manual, atol=1e-6)

## 8. El problema autorregresivo y la máscara causal

Para predecir el siguiente token, la posición \(i\) no puede utilizar posiciones \(j>i\).

Definimos

\[
M_{ij}=
\begin{cases}
0,&j\le i,\\
-\infty,&j>i.
\end{cases}
\]

La máscara se aplica **antes** de softmax. Así, \(e^{-\infty}=0\) y las conexiones futuras reciben peso exactamente nulo.

In [ ]:
T = len(frase)

# TODO 9: máscara booleana True por encima de la diagonal.
mask = ...

scores_masked = scores.masked_fill(mask, float('-inf'))

# TODO 10: aplica softmax para obtener la atención causal.
A = ...

print(A[0])
print('máximo peso prohibido:',
      torch.triu(A[0], diagonal=1).abs().max().item())

assert torch.allclose(torch.triu(A[0], diagonal=1),
                      torch.zeros_like(A[0]),
                      atol=1e-7)
assert torch.allclose(A.sum(dim=-1),
                      torch.ones_like(A.sum(dim=-1)),
                      atol=1e-6)

In [ ]:
plt.figure(figsize=(6,5))
plt.imshow(A[0].detach().numpy(), vmin=0, vmax=1)
plt.xticks(range(T), frase, rotation=45)
plt.yticks(range(T), frase)
plt.xlabel('posición que envía información (key/value)')
plt.ylabel('posición que recibe información (query)')
plt.title('Matriz de self-attention causal')
plt.colorbar(label='peso de atención')
plt.tight_layout()
plt.show()

## 9. Una cabeza causal completa, sin capas de atención preconstruidas

Ahora encapsulamos exactamente la operación

\[
\operatorname{CausalAttention}(X)
=
\mathrm{softmax}\!\left(
\frac{QK^T}{\sqrt{d_k}}+M
\right)V.
\]

No usamos `nn.MultiheadAttention`.

In [ ]:
class SelfAttentionCausal(nn.Module):
    def __init__(self, d_model, d_k, d_v):
        super().__init__()
        self.Wq = nn.Linear(d_model, d_k, bias=False)
        self.Wk = nn.Linear(d_model, d_k, bias=False)
        self.Wv = nn.Linear(d_model, d_v, bias=False)
        self.scale = math.sqrt(d_k)

    def forward(self, X, return_attention=False):
        # TODO 11: Q, K, Vv.
        Q = ...
        K = ...
        Vv = ...

        # TODO 12: scores escalados.
        scores = ...

        T = X.shape[1]
        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool, device=X.device),
            diagonal=1
        )

        # TODO 13: enmascara, normaliza y agrega values.
        scores = ...
        A = ...
        Z = ...

        return (Z, A) if return_attention else Z

## 10. Test de causalidad: cambiar el futuro no puede modificar el pasado

Esta comprobación es más fuerte que mirar la matriz triangular.

Si alteramos el último token de la secuencia, las salidas de posiciones anteriores deben permanecer idénticas:

\[
X_{T}\to X'_{T}
\quad\Longrightarrow\quad
z_i=z'_i\quad\text{para }i<T.
\]

In [ ]:
att = SelfAttentionCausal(d_model=8, d_k=8, d_v=8)

E = nn.Embedding(len(tokens), 8)
ids1 = torch.tensor([[token_a_id[t] for t in
                      ['la','masa','curva','el','espacio']]])
ids2 = ids1.clone()
ids2[0, -1] = token_a_id['tiempo']

X1, X2 = E(ids1), E(ids2)
Z1, A1 = att(X1, return_attention=True)
Z2, A2 = att(X2, return_attention=True)

# TODO 14: compara todas las posiciones anteriores a la última.
error_pasado = ...

print('error máximo en el pasado:', error_pasado)
assert error_pasado < 1e-6

## 11. La atención como grafo dinámico

La matriz \(A(X)\) puede interpretarse como los pesos de una red dirigida:

\[
A_{ij}>0
\quad\Longleftrightarrow\quad
j\to i.
\]

Pero no es una matriz de adyacencia fija. Como

\[
A(X)=
\mathrm{softmax}\!\left(
\frac{(XW_Q)(XW_K)^T}{\sqrt{d_k}}+M
\right),
\]

al cambiar \(X\) cambia, en general, la red de pesos.

Compara dos secuencias con los **mismos parámetros** de atención.

In [ ]:
seq_a = ['la','masa','curva','el','espacio']
seq_b = ['la','luz','curva','el','tiempo']

Ia = torch.tensor([[token_a_id[t] for t in seq_a]])
Ib = torch.tensor([[token_a_id[t] for t in seq_b]])

with torch.no_grad():
    _, Aa = att(E(Ia), return_attention=True)
    _, Ab = att(E(Ib), return_attention=True)

# TODO 15: calcula una medida simple de cuánto cambian ambas matrices.
cambio = ...

print('cambio máximo |A(X)-A(X\')| =', cambio)

fig = plt.figure(figsize=(6,4))
plt.imshow((Aa[0]-Ab[0]).abs().numpy())
plt.title('|A(X) - A(X\')|')
plt.xlabel('key/value')
plt.ylabel('query')
plt.colorbar()
plt.tight_layout()
plt.show()

# Problema final abierto — Recuperación asociativa con una cabeza de atención

Queremos construir un problema donde la utilidad de \(Q\), \(K\) y \(V\) pueda verse directamente.

Cada ejemplo contiene tres pares clave–valor, uno para cada clave \(A,B,C\), con valor binario. Los tres pares aparecen en **orden aleatorio** y al final hay una consulta.

Ejemplo:

\[
[B1,\ A0,\ C1,\ QA]\longrightarrow 0.
\]

La consulta `QA` pide recuperar el valor asociado a la clave \(A\), que en ese ejemplo es \(0\).

Otro:

\[
[A1,\ C0,\ B0,\ QC]\longrightarrow 0.
\]

### Tu objetivo

Construye un modelo que aprenda esta operación mediante **una única cabeza de self-attention causal programada por ti**.

### Entrega

1. Genera al menos 2400 ejemplos de entrenamiento y 600 de validación.
2. Usa el vocabulario `A0,A1,B0,B1,C0,C1,QA,QB,QC`.
3. Baraja las tres fichas clave–valor en cada ejemplo. La consulta queda en la última posición.
4. Construye
   \[
   \text{embedding}\to\text{self-attention causal}\to
   \text{representación de la última posición}\to\text{clasificador binario}.
   \]
5. **No uses** `nn.MultiheadAttention`, `nn.Transformer` ni `nn.TransformerEncoder`.
6. Reporta cross-entropy y accuracy de validación.
7. Para al menos cinco ejemplos, muestra la última fila de \(A(X)\) y señala a qué ficha clave–valor presta más atención la query.
8. Permuta las tres primeras fichas de un mismo ejemplo. Comprueba si cambia la predicción y razona qué papel desempeña la ausencia de codificación posicional.
9. Compara contra un baseline
   \[
   \text{embedding}\to\text{promedio uniforme}\to\text{clasificador lineal}.
   \]
10. Interpreta qué deberían aprender conceptualmente \(Q\), \(K\) y \(V\):
    - ¿qué información necesita una query `QA`?
    - ¿qué debe hacer reconocible a una key `A0` o `A1`?
    - ¿qué debe transportar el value de `A0` frente al de `A1`?
11. Cambia el problema a cuatro claves \(A,B,C,D\). ¿Sigue funcionando?
12. Explica por qué este experimento puede interpretarse como una **búsqueda suave en un diccionario** y, simultáneamente, como *message passing* sobre un grafo dinámico.

In [ ]:
# RETO A — genera los datos
KEYS = ['A','B','C']
...

# RETO B — construye el modelo de recuperación
...

# RETO C — entrena y evalúa
...

# RETO D — inspecciona mapas de atención
...

# RETO E — permutaciones y baseline uniforme
...

# RETO F — amplía a cuatro claves
...